# Pemodelan Sistem Dinamik Persediaan Beras Rumah Tangga akibat Pemakaian dan Pembelian

Notebook ini berisi seluruh program pada laporan. Jalankan sel dari atas ke bawah (Runtime > Run all).

In [ ]:
!pip -q install gekko

## 2.1 Hanya ada pembelian

In [ ]:
# Program 1: Simulasi Stok Beras Hanya dengan Pembelian (ODEINT)
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# model stok beras: hanya ada pembelian, pemakaian diabaikan
def stok(S, t, kb, ub):
    dSdt = kb * ub                # laju penambahan stok (kg/hari)
    return dSdt

# waktu simulasi 30 hari, setiap 0.1 hari
ts = np.linspace(0, 30, 301)

# operasi pembelian
kb = 0.1                          # koefisien pembelian (kg/hari per %)
ub = np.zeros(301)                # ub = intensitas pembelian (%)
for hari in [3, 11, 19, 27]:      # belanja selama 1 hari penuh
    ub[hari*10:(hari+1)*10] = 100.0

# kondisi awal stok
S0 = 5.0

# penyimpanan hasil
z = np.zeros(301)
z[0] = S0

# simulasi dengan ODEINT
for i in range(300):
    y = odeint(stok, S0, [0, 0.1], args=(kb, ub[i]))
    S0 = y[-1, 0]                 # ambil titik terakhir
    z[i+1] = S0                   # simpan stok untuk plot

# plot hasil
plt.figure()
plt.subplot(2, 1, 1)
plt.plot(ts, z, 'b-')
plt.ylabel('Stok Beras $S(t)$, (kg)')
plt.grid(alpha=0.3)
plt.subplot(2, 1, 2)
plt.plot(ts, ub, 'r--')
plt.ylabel('Pembelian $u_b(t)$, (%)')
plt.xlabel('$t$, (hari)')
plt.show()

## 2.2 Pembelian dan pemakaian

In [ ]:
# Program 2: Simulasi Stok Beras dengan Pembelian dan Pemakaian (ODEINT)
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# model stok beras: pembelian dan pemakaian
def stok(S, t, kb, kc, ub, uc):
    dSdt = kb * ub - kc * uc      # laju pembelian - laju pemakaian
    return dSdt

# waktu simulasi 30 hari, setiap 0.1 hari
ts = np.linspace(0, 30, 301)

# operasi pembelian dan pemakaian
kb = 0.1                          # koefisien pembelian (kg/hari per %)
kc = 0.012                        # koefisien pemakaian (kg/hari per %)
ub = np.zeros(301)                # ub = intensitas pembelian (%)
for hari in [3, 11, 19, 27]:
    ub[hari*10:(hari+1)*10] = 100.0
uc = np.full(301, 100.0)          # uc = intensitas pemakaian (%) -> 1.2 kg/hari

# kondisi awal stok
S0 = 5.0

# penyimpanan hasil
z = np.zeros(301)
z[0] = S0

# simulasi dengan ODEINT
for i in range(300):
    y = odeint(stok, S0, [0, 0.1], args=(kb, kc, ub[i], uc[i]))
    S0 = y[-1, 0]
    z[i+1] = S0

# plot hasil
plt.figure()
plt.subplot(2, 1, 1)
plt.plot(ts, z, 'b-')
plt.ylabel('Stok Beras $S(t)$, (kg)')
plt.grid(alpha=0.3)
plt.subplot(2, 1, 2)
plt.plot(ts, ub, 'r--', label='$u_b$')
plt.plot(ts, uc, 'g--', label='$u_c$')
plt.ylabel('Kontrol $u(t)$, (%)')
plt.xlabel('$t$, (hari)')
plt.legend(frameon=False)
plt.show()

## 3.1 Contoh 1

In [ ]:
# Program 3: Program Untuk Contoh 1 (ODEINT)
from scipy.integrate import odeint
import numpy as np
import matplotlib.pyplot as plt

# fungsi yang mengembalikan dS/dt
def model(y, t):
    k = 0.1
    dydt = -k * y
    return dydt

# kondisi awal
y0 = 10

# titik waktu
t = np.linspace(0, 20)

# selesaikan ODE
y = odeint(model, y0, t)

# plot hasil
plt.plot(t, y)
plt.xlabel('$t$, (hari)')
plt.ylabel('$y(t)$, (kg)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Program 4: Program Untuk Contoh 1 (GEKKO)
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

m = GEKKO()                       # membuat model GEKKO
k = 0.1                           # konstanta
y = m.Var(10.0)                   # membuat variabel GEKKO
m.Equation(y.dt() == -k * y)      # membuat persamaan GEKKO
m.time = np.linspace(0, 20)       # titik waktu

# selesaikan ODE
m.options.IMODE = 4
m.options.NODES = 3
m.solve(disp=False)

# plot hasil
plt.plot(m.time, y.value)
plt.xlabel('$t$, (hari)')
plt.ylabel('$y(t)$, (kg)')
plt.grid(alpha=0.3)
plt.show()

## 3.2 Contoh 2

In [ ]:
# Program 5: Program Untuk Contoh 2 (ODEINT)
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# fungsi yang mengembalikan dy/dt
def model(y, t, k):
    dydt = -k * y
    return dydt

# kondisi awal
y0 = 20

# titik waktu
t = np.linspace(0, 20)

# selesaikan ODE untuk tiga nilai k
k = 0.05
y1 = odeint(model, y0, t, args=(k,))
k = 0.1
y2 = odeint(model, y0, t, args=(k,))
k = 0.2
y3 = odeint(model, y0, t, args=(k,))

# plot hasil
plt.plot(t, y1, 'r-',  label='$k=0.05$')
plt.plot(t, y2, 'b--', label='$k=0.1$')
plt.plot(t, y3, 'g:',  label='$k=0.2$')
plt.xlabel('$t$, (hari)')
plt.ylabel('$y(t)$, (kg)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Program 6: Program Untuk Contoh 2 (GEKKO)
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

m = GEKKO()                       # membuat model GEKKO
k = m.Param()                     # konstanta
y = m.Var(20.0)                   # membuat variabel GEKKO
m.Equation(y.dt() == -k * y)      # membuat persamaan GEKKO
m.time = np.linspace(0, 20)       # titik waktu

# selesaikan ODE dan plot
m.options.IMODE = 4
m.options.NODES = 3
m.options.TIME_SHIFT = 0

k.value = 0.05
m.solve(disp=False)
plt.plot(m.time, y.value, 'r-', linewidth=2, label='$k=0.05$')

k.value = 0.1
m.solve(disp=False)
plt.plot(m.time, y.value, 'b--', linewidth=2, label='$k=0.1$')

k.value = 0.2
m.solve(disp=False)
plt.plot(m.time, y.value, 'g:', linewidth=2, label='$k=0.2$')

plt.xlabel('$t$, (hari)')
plt.ylabel('$y(t)$, (kg)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3.3 Contoh 3

In [ ]:
# Program 7: Program Untuk Contoh 3 (ODEINT)
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# fungsi yang mengembalikan dy/dt
def model(y, t):
    # u naik dari 0 ke 20 pada t = 10
    if t < 10.0:
        u = 0
    else:
        u = 20
    dydt = (-y + u) / 5.0
    return dydt

# kondisi awal
y0 = 10

# titik waktu
t = np.linspace(0, 40, 1000)

# selesaikan ODE
y = odeint(model, y0, t)

# plot hasil
plt.plot(t, y, 'r-', label='Output, $y(t)$')
plt.plot([0, 10, 10, 40], [0, 0, 20, 20], 'b-', label='Input, $u(t)$')
plt.ylabel('$y(t)$ dan $u(t)$, (kg)')
plt.xlabel('$t$, (hari)')
plt.legend(loc='best', frameon=False)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Program 8: Program Untuk Contoh 3 (GEKKO)
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

m = GEKKO()                       # membuat model GEKKO
m.time = np.linspace(0, 40, 401)  # titik waktu

# membuat parameter GEKKO (step 0 ke 20 pada t = 10)
ustep = np.zeros(401)
ustep[100:] = 20.0
u = m.Param(value=ustep)

y = m.Var(10.0)                   # membuat variabel GEKKO
m.Equation(5 * y.dt() == -y + u)  # membuat persamaan GEKKO

# selesaikan ODE
m.options.IMODE = 4
m.options.NODES = 3
m.solve(disp=False)

# plot hasil
plt.plot(m.time, y.value, 'r-', label='Output ($y(t)$)')
plt.plot(m.time, u.value, 'b-', label='Input ($u(t)$)')
plt.xlabel('$t$, (hari)')
plt.ylabel('nilai (kg)')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.show()

## 3.4 Contoh 4

In [ ]:
# Program 9: Program Untuk Contoh 4 (ODEINT)
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# fungsi yang mengembalikan dz/dt
def model(z, t):
    dxdt = 3.0 * np.exp(-t)
    dydt = -z[1] + 3
    dzdt = [dxdt, dydt]
    return dzdt

# kondisi awal
z0 = [0, 0]

# titik waktu
t = np.linspace(0, 5)

# selesaikan ODE
z = odeint(model, z0, t)

# plot hasil
plt.plot(t, z[:, 0], 'b-',  label=r'$\frac{dx}{dt}=3\,\exp(-t)$')
plt.plot(t, z[:, 1], 'r--', label=r'$\frac{dy}{dt}=-y+3$')
plt.ylabel('$x(t),\\; y(t)$, (kg)')
plt.xlabel('$t$, (hari)')
plt.legend(loc='best', frameon=False)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Program 10: Program Untuk Contoh 4 (GEKKO)
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

m = GEKKO()                       # membuat model GEKKO
m.time = np.linspace(0, 5)        # titik waktu

t = m.Var(0.0)                    # membuat waktu sebagai variabel GEKKO
x = m.Var(0.0)                    # membuat variabel GEKKO x
y = m.Var(0.0)                    # membuat variabel GEKKO y

# membuat persamaan GEKKO
m.Equation(t.dt() == 1)
m.Equation(x.dt() == 3 * m.exp(-t))
m.Equation(y.dt() == 3 - y)

# selesaikan ODE
m.options.IMODE = 4
m.options.NODES = 3
m.solve(disp=False)

# plot hasil
plt.plot(t.value, x.value, 'b-',  label=r'$\frac{dx}{dt}=3\,e^{-t}$')
plt.plot(t.value, y.value, 'r--', label=r'$\frac{dy}{dt}=-y+3$')
plt.ylabel('respon (kg)')
plt.xlabel('waktu (hari)')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.show()

## 3.5 Contoh 5

In [ ]:
# Program 11: Program Untuk Contoh 5 (ODEINT)
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# fungsi yang mengembalikan dz/dt
def model(z, t, u):
    x = z[0]
    y = z[1]
    dxdt = (-x + u) / 2.0
    dydt = (-y + x) / 5.0
    dzdt = [dxdt, dydt]
    return dzdt

# kondisi awal
z0 = [0, 0]

# jumlah titik waktu
n = 401

# titik waktu
t = np.linspace(0, 40, n)

# input step
u = np.zeros(n)
# berubah menjadi 20 pada t = 5.0
u[50:] = 20.0

# penyimpanan solusi
x = np.empty_like(t)
y = np.empty_like(t)
# catat kondisi awal
x[0] = z0[0]
y[0] = z0[1]

# selesaikan ODE
for i in range(1, n):
    # rentang waktu untuk langkah berikutnya
    tspan = [t[i-1], t[i]]
    # selesaikan langkah berikutnya
    z = odeint(model, z0, tspan, args=(u[i-1],))
    # simpan solusi untuk plot
    x[i] = z[1][0]
    y[i] = z[1][1]
    # kondisi awal berikutnya
    z0 = z[1]

# plot hasil
plt.plot(t, u, 'g:',  label='$u(t)$')
plt.plot(t, x, 'b-',  label='$x(t)$')
plt.plot(t, y, 'r--', label='$y(t)$')
plt.ylabel('$u(t),\\; x(t),\\; y(t)$, (kg)')
plt.xlabel('$t$, (hari)')
plt.legend(loc='best', frameon=False)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Program 12: Program Untuk Contoh 5 (GEKKO)
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

m = GEKKO()                       # membuat model GEKKO
m.time = np.linspace(0, 40, 401)  # titik waktu

# membuat parameter GEKKO (step 0 ke 20 pada t = 5)
ustep = np.zeros(401)
ustep[50:] = 20.0
u = m.Param(value=ustep)

# membuat variabel GEKKO
x = m.Var(0.0)
y = m.Var(0.0)

# membuat persamaan GEKKO
m.Equation(2 * x.dt() == -x + u)
m.Equation(5 * y.dt() == -y + x)

# selesaikan ODE
m.options.IMODE = 4
m.options.NODES = 3
m.solve(disp=False)

# plot hasil
plt.plot(m.time, u.value, 'g:',  label='$u(t)$')
plt.plot(m.time, x.value, 'b-',  label='$x(t)$')
plt.plot(m.time, y.value, 'r--', label='$y(t)$')
plt.xlabel('$t$, (hari)')
plt.ylabel('nilai (kg)')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.show()

## 4. Studi kasus: persediaan beras dengan kebijakan pembelian dan penghematan

In [ ]:
# Program 13: Simulasi Persediaan Beras Rumah Tangga dengan Kebijakan Pembelian (ODEINT)
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# ---------------- parameter ----------------
n_orang = 4          # jumlah anggota keluarga
c_orang = 0.3        # konsumsi beras (kg/orang/hari)
kc = n_orang * c_orang / 100.0   # koefisien pemakaian (kg/hari per %) = 0.012
kb = 0.1             # koefisien pembelian (kg/hari per %)
ks = 0.002           # laju susut/rusak beras (1/hari)
ROP = 3.0            # titik pemesanan ulang (kg)
lama_beli = 1.0      # lama satu kali belanja (hari) -> 1 x 10 kg

# ---------------- model ----------------
def beras(S, t, ub, uc):
    pakai = kc * uc if S > 0 else 0.0     # tidak bisa memakai jika stok habis
    dSdt = kb * ub - pakai - ks * S
    return dSdt

# ---------------- simulasi satu skenario ----------------
def simulasi(u_hemat, T=60, dt=0.1, S0=10.0):
    N = int(T/dt)
    ts = np.linspace(0, T, N+1)
    S  = np.zeros(N+1); S[0] = S0
    ub = np.zeros(N+1); uc = np.zeros(N+1)
    sedang_beli, akhir_beli = False, 0.0
    jumlah_beli = 0
    for i in range(N):
        # kebijakan: beli 1 hari penuh jika stok <= ROP
        if sedang_beli and ts[i] >= akhir_beli:
            sedang_beli = False
        if (not sedang_beli) and S[i] <= ROP:
            sedang_beli, akhir_beli = True, ts[i] + lama_beli
            jumlah_beli += 1
        ub[i] = 100.0 if sedang_beli else 0.0
        # pemakaian: normal 100%, naik 150% saat ada acara keluarga (hari 30-37)
        faktor = 1.5 if 30 <= ts[i] < 37 else 1.0
        uc[i] = 100.0 * faktor * (1 - u_hemat)
        y = odeint(beras, S[i], [0, dt], args=(ub[i], uc[i]))
        S[i+1] = max(y[-1, 0], 0.0)
    ub[N], uc[N] = ub[N-1], uc[N-1]
    return ts, S, ub, uc, jumlah_beli

# ---------------- tiga skenario penghematan ----------------
hasil = {}
for u in [0.0, 0.1, 0.2]:
    hasil[u] = simulasi(u)
    ts, S, ub, uc, nb = hasil[u]
    total_beli = np.sum(kb * ub) * 0.1
    hari_habis = np.sum(S <= 0) * 0.1
    print(f'u = {u:.1f} | beli {nb} kali | total {total_beli:.0f} kg | '
          f'stok habis {hari_habis:.1f} hari | stok akhir {S[-1]:.2f} kg')

# ---------------- plot hasil ----------------
warna = {0.0: 'b-', 0.1: 'g--', 0.2: 'r:'}
plt.figure(figsize=(9, 6))
plt.subplot(2, 1, 1)
for u in hasil:
    plt.plot(hasil[u][0], hasil[u][1], warna[u], label=f'$u={u}$')
plt.axhline(ROP, color='gray', linestyle='--', linewidth=1)
plt.text(1, ROP - 0.6, 'ROP = 3 kg', color='gray')
plt.axvspan(30, 37, color='orange', alpha=0.15)
plt.ylabel('Stok Beras $S(t)$, (kg)')
plt.legend(frameon=False, ncol=3, loc='lower center', bbox_to_anchor=(0.5, 1.0))
plt.grid(alpha=0.3)

plt.subplot(2, 1, 2)
plt.plot(hasil[0.0][0], hasil[0.0][3], 'g-',  label='$u_c$ ($u=0$)')
plt.plot(hasil[0.0][0], hasil[0.0][2], 'r--', label='$u_b$ ($u=0$)')
plt.axvspan(30, 37, color='orange', alpha=0.15)
plt.ylabel('Kontrol $u(t)$, (%)')
plt.xlabel('$t$, (hari)')
plt.legend(frameon=False, loc='upper right')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Program 14: Simulasi Persediaan Beras Rumah Tangga (GEKKO)
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

# parameter sama dengan Program 13
kc, kb, ks = 0.012, 0.1, 0.002

# jadwal pembelian ub dan pemakaian uc skenario u = 0 diambil dari Program 13
ts, S_odeint, ub, uc, _ = hasil[0.0]

m = GEKKO()
m.time = ts
UB = m.Param(value=ub)           # intensitas pembelian (%)
UC = m.Param(value=uc)           # intensitas pemakaian (%)
S  = m.Var(10.0)                 # stok beras (kg)

# persamaan stok beras
m.Equation(S.dt() == kb * UB - kc * UC - ks * S)

# selesaikan ODE
m.options.IMODE = 4
m.options.NODES = 3
m.solve(disp=False)

# bandingkan dengan hasil ODEINT
selisih = np.max(np.abs(np.array(S.value) - S_odeint))
print(f'Selisih maksimum GEKKO vs ODEINT = {selisih:.3f} kg')

plt.plot(ts, S_odeint, 'b-', label='ODEINT')
plt.plot(m.time, S.value, 'r--', label='GEKKO')
plt.xlabel('$t$, (hari)')
plt.ylabel('Stok Beras $S(t)$, (kg)')
plt.legend(frameon=False)
plt.grid(alpha=0.3)
plt.show()